# Precomputed MLP Forward Return

Load precomputed LOB features and forward-return labels from `data/orderbook_feature_return_parquet`, infer the feature set from the parquet schema, then run rolling time-series validation with the streaming `TorchAdapter` and a configurable PyTorch MLP.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import re
import sys
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import polars as pl
import torch
from matplotlib import pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.data import DataSource, DateFrame, Raw, expand_dates
from tools.filters import intraday_time, level_taken, tight_spread, trade_size
from tools.model import TorchAdapter
from tools.pipeline import Pipeline
from tools.score import get_pinball, get_quantile_pnl, get_unit_pnl, rmse
from tools.transform import Standardizer

In [3]:
def divide_dates(*args):
    dates = []
    for i in range(1, len(args)):
        dates.append(
            expand_dates(
                f"{args[i - 1]}-{args[i]}",
                end_date=False if i < len(args) - 1 else True,
            )
        )
    return dates

In [4]:
# Data
PROD = "ES"
ROLLING_DATES = divide_dates(20260323, 20260410, 20260425, 20260510, 20260524)
TEST_DATES = expand_dates("20260525-20260529")
L2_DEPTH = 5
MODEL_BATCH_SIZE = 8_192
POLARS_ENGINE = "streaming"
FEATURE_RETURN_PATH = str(
    ROOT
    / f"data/orderbook_feature_return_parquet/{{prod}}M6_{{d}}_{{tag}}_{{prod_s}}_full_day_l2_d{L2_DEPTH}_features_return.parquet"
)
REGULAR_HOURS_START = "09:30"
REGULAR_HOURS_END = "16:00"
REGULAR_HOURS_TZ = "America/New_York"

# Forward-return target column already present in FEATURE_RETURN_PATH files.
TARGET = "forward_mid_return_bps"
TEST_PNL_THRESHOLD = 0.2

# MLP/search knobs
SEED = 7
SAMPLER = "random"
N_TRIALS = 20
EPOCHS = 5
DEVICE = "cuda"
QUANTILES = [0.1, 0.5, 0.9]
MEDIAN_IDX = QUANTILES.index(0.5)

DEFAULT_MLP_PARAMS = {
    "hidden_layers": 2,
    "hidden_units": [128, 64],
    "activation": "silu",
    "dropout": 0.05,
    "lr": 1e-3,
    "weight_decay": 1e-4,
}
TUNE_ARCHITECTURE = True
HIDDEN_LAYER_CHOICES = [1, 2, 3]
HIDDEN_UNITS_CHOICES = [32, 64, 128, 256]
ACTIVATION_CHOICES = ["silu", "relu", "gelu"]
DROPOUT_CHOICES = [0.0, 0.05, 0.1]
LEARNING_RATE_RANGE = (1e-4, 3e-3)
WEIGHT_DECAY_RANGE = (1e-6, 1e-2)

UNDEF_PRICE = 9_223_372_036_854_775_807
TICKSIZE = 250000000

np.random.seed(SEED)

def median_quantile(score):
    def wrapped(y_true, y_pred, ctx=None, **kwargs):
        y_pred = np.asarray(y_pred)
        if y_pred.ndim == 2:
            y_pred = y_pred[:, MEDIAN_IDX]
        return score(y_true, y_pred, ctx, **kwargs)

    wrapped.__name__ = f"median_{getattr(score, '__name__', 'score')}"
    return wrapped


torch.manual_seed(SEED)
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' requested, but torch.cuda.is_available() is False.")

In [5]:
BOOK_COL_RE = re.compile(r"^(?:bid|ask)_(?:px|sz|ct)_\d+$")
SCHEMA_NON_FEATURE_COLS = {
    "date",
    "nature",
    "ts_event",
    "ts_recv",
    "symbol",
    "instrument_id",
    "row_nr",
    "sequence",
    "publisher_id",
    "trade_px",
    "trade_sz",
    "trade_side",
}

def infer_features_from_schema(schema: pl.Schema, target: str = TARGET) -> list[str]:
    features = []
    for col in schema.names():
        if col == target or col in SCHEMA_NON_FEATURE_COLS or BOOK_COL_RE.match(col):
            continue
        features.append(col)
    if not features:
        raise ValueError("no feature columns inferred from parquet schema")
    return features

FEATURE_SCHEMA_PATH, _ = Raw.resolve_path(ROLLING_DATES[0][0], PROD, FEATURE_RETURN_PATH)
FEATURE_SCHEMA = pl.scan_parquet(FEATURE_SCHEMA_PATH).collect_schema()
FEATURES = infer_features_from_schema(FEATURE_SCHEMA)
# FEATURES = ["weighted_price_sz2"]
META_COLS = [col for col in FEATURE_SCHEMA.names() if col not in FEATURES and col != TARGET]
LOAD_COLS = list(dict.fromkeys([*META_COLS, *FEATURES, TARGET]))

FEATURES

['imb_d1',
 'imb_d3',
 'imb_d5',
 'weighted_price_sz2',
 'weighted_price_sz5',
 'weighted_price_sz10',
 'trade_momentum_hl1s',
 'trade_momentum_hl10s',
 'trade_momentum_hl30s',
 'trade_momentum_hl120s']

In [6]:
VALID_ROWS = (
    (pl.col("bid_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") > pl.col("bid_px_0"))
    & pl.col(TARGET).is_not_null()
    & pl.all_horizontal([pl.col(c).is_finite() for c in FEATURES])
)
REGULAR_HOURS = intraday_time(REGULAR_HOURS_START, REGULAR_HOURS_END, timezone=REGULAR_HOURS_TZ)
TIGHT_SPREAD = tight_spread(TICKSIZE)
VALID_REGULAR_ROWS = VALID_ROWS & REGULAR_HOURS & TIGHT_SPREAD
TRAIN_ROWS = VALID_REGULAR_ROWS & (level_taken() | trade_size(0.3))

REGULAR_HOURS

<Expr ['[([(col("ts_event").dt.convert…'] at 0x76BE1D316300>

In [7]:
def load_feature_return_date(day: str, prod: str = PROD) -> DateFrame:
    return Raw.load_date(day, prod, path=FEATURE_RETURN_PATH, cols=LOAD_COLS)


def regular_loader(dates: list[str]) -> list[DateFrame]:
    return [load_feature_return_date(day) for day in dates]

In [8]:
FEATURE_TEST_SCORE = get_unit_pnl(0.3)
FEATURE_TEST_SCORE_DESCENDING = True

test_date_src = DataSource(
    dates=TEST_DATES,
    loader=regular_loader,
    target=TARGET,
    features=FEATURES,
    filters=(VALID_REGULAR_ROWS,),
    polars_engine=POLARS_ENGINE,
)

feature_test_states = dict.fromkeys(FEATURES)
feature_test_rows = 0
for x, y_true, ctx in test_date_src.batches(MODEL_BATCH_SIZE):
    feature_test_rows += int(ctx["n"])
    for idx, feature in enumerate(FEATURES):
        feature_test_states[feature] = FEATURE_TEST_SCORE(
            y_true,
            x[:, idx],
            ctx,
            combine_with=feature_test_states[feature],
        )

feature_test_scores = (
    pl.DataFrame(
        [
            {
                "feature": feature,
                "score": getattr(FEATURE_TEST_SCORE, "__name__", "score"),
                "test_score": float(state),
                "score_n": int(getattr(state, "n", 0)),
                "rows": feature_test_rows,
            }
            for feature, state in feature_test_states.items()
            if state is not None
        ]
    )
    .sort("test_score", descending=FEATURE_TEST_SCORE_DESCENDING)
)

feature_test_scores

Loading data: 25.6Mrow [00:09, 2.57Mrow/s]


feature,score,test_score,score_n,rows
str,str,f64,i64,i64
"""weighted_price_sz2""","""unit_pnl_0.3""",0.972791,217,25573459
"""imb_d5""","""unit_pnl_0.3""",0.120021,16491404,25573459
"""imb_d3""","""unit_pnl_0.3""",0.115116,18452295,25573459
"""imb_d1""","""unit_pnl_0.3""",0.106025,21833709,25573459
"""trade_momentum_hl120s""","""unit_pnl_0.3""",0.069664,6316273,25573459
"""trade_momentum_hl1s""","""unit_pnl_0.3""",0.003674,21038404,25573459
"""trade_momentum_hl10s""","""unit_pnl_0.3""",-0.014114,16049639,25573459
"""trade_momentum_hl30s""","""unit_pnl_0.3""",-0.020354,12234031,25573459
"""weighted_price_sz5""","""unit_pnl_0.3""",-0.439122,858,25573459


The architecture is controlled by `hidden_layers` and either one `hidden_units` value, a `hidden_units` list, or per-layer `hidden_units_l1`, `hidden_units_l2`, ... values. Set `TUNE_ARCHITECTURE = False` and `N_TRIALS = 1` to train only `DEFAULT_MLP_PARAMS`.

In [9]:
def activation_layer(name: str) -> torch.nn.Module:
    name = name.lower()
    if name == "relu":
        return torch.nn.ReLU()
    if name == "gelu":
        return torch.nn.GELU()
    if name == "silu":
        return torch.nn.SiLU()
    if name == "tanh":
        return torch.nn.Tanh()
    raise ValueError(f"unsupported activation: {name}")


def hidden_sizes_from_params(params: dict[str, object]) -> list[int]:
    hidden_layers = int(params.get("hidden_layers", DEFAULT_MLP_PARAMS["hidden_layers"]))
    if hidden_layers < 0:
        raise ValueError("hidden_layers must be non-negative")

    units = params.get("hidden_units")
    if isinstance(units, str):
        sizes = [int(part.strip()) for part in units.split(",") if part.strip()]
    elif isinstance(units, Sequence):
        sizes = [int(unit) for unit in units]
    elif units is not None:
        sizes = [int(units)] * hidden_layers
    else:
        default_units = DEFAULT_MLP_PARAMS["hidden_units"]
        fallback = default_units[0] if isinstance(default_units, Sequence) else default_units
        sizes = [int(params.get(f"hidden_units_l{i + 1}", fallback)) for i in range(hidden_layers)]

    if len(sizes) < hidden_layers:
        fill = sizes[-1] if sizes else int(DEFAULT_MLP_PARAMS["hidden_units"][0])
        sizes.extend([fill] * (hidden_layers - len(sizes)))
    sizes = sizes[:hidden_layers]
    if any(width <= 0 for width in sizes):
        raise ValueError(f"hidden layer widths must be positive: {sizes}")
    return sizes


def torch_pinball_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    pred = y_pred.float()
    if pred.ndim == 1:
        pred = pred[:, None]
    y = y_true.float().reshape(-1, 1)
    q = torch.as_tensor(QUANTILES, dtype=pred.dtype, device=pred.device)
    err = y - pred
    return torch.maximum(q * err, (q - 1.0) * err).mean()


def build_mlp(params: dict[str, object]) -> torch.nn.Module:
    torch.manual_seed(int(params.get("seed", SEED)))
    hidden_sizes = hidden_sizes_from_params(params)
    activation = str(params.get("activation", DEFAULT_MLP_PARAMS["activation"]))
    dropout = float(params.get("dropout", DEFAULT_MLP_PARAMS["dropout"]))

    layers: list[torch.nn.Module] = []
    in_features = len(FEATURES)
    for width in hidden_sizes:
        layers.append(torch.nn.Linear(in_features, width))
        layers.append(activation_layer(activation))
        if dropout > 0.0:
            layers.append(torch.nn.Dropout(dropout))
        in_features = width
    layers.append(torch.nn.Linear(in_features, len(QUANTILES)))

    model = torch.nn.Sequential(*layers)
    setattr(model, "_hidden_sizes", hidden_sizes)
    setattr(model, "_quantiles", tuple(QUANTILES))
    return model


def build_optimizer(parameters, params: dict[str, object]):
    return torch.optim.AdamW(
        parameters,
        lr=float(params.get("lr", DEFAULT_MLP_PARAMS["lr"])),
        weight_decay=float(params.get("weight_decay", DEFAULT_MLP_PARAMS["weight_decay"])),
    )


def mlp_search_space(trial) -> dict[str, object]:
    if not TUNE_ARCHITECTURE:
        return dict(DEFAULT_MLP_PARAMS)

    hidden_layers = trial.suggest_categorical("hidden_layers", HIDDEN_LAYER_CHOICES)
    params: dict[str, object] = {
        "hidden_layers": hidden_layers,
        "activation": trial.suggest_categorical("activation", ACTIVATION_CHOICES),
        "dropout": trial.suggest_categorical("dropout", DROPOUT_CHOICES),
        "lr": trial.suggest_float("lr", *LEARNING_RATE_RANGE, log=True),
        "weight_decay": trial.suggest_float("weight_decay", *WEIGHT_DECAY_RANGE, log=True),
        "seed": SEED,
    }
    for layer_idx in range(1, int(hidden_layers) + 1):
        params[f"hidden_units_l{layer_idx}"] = trial.suggest_categorical(
            f"hidden_units_l{layer_idx}",
            HIDDEN_UNITS_CHOICES,
        )
    return params


hidden_sizes_from_params(DEFAULT_MLP_PARAMS)

[128, 64]

In [10]:
pipeline = Pipeline(
    rolling_dates=ROLLING_DATES,
    test_dates=TEST_DATES,
    adapter=TorchAdapter(
        module_builder=build_mlp,
        loss_fn=torch_pinball_loss,
        optimizer_builder=build_optimizer,
        epochs=EPOCHS,
        batch_size=MODEL_BATCH_SIZE,
        device=DEVICE,
        streaming=True,
    ),
    target=TARGET,
    features=FEATURES,
    data_loader=regular_loader,
    search_space=mlp_search_space,
    val_score=get_pinball(QUANTILES),
    transform=Standardizer(FEATURES),
    train_filters=(TRAIN_ROWS,),
    val_filters=(VALID_REGULAR_ROWS,),
    test_filters=(VALID_REGULAR_ROWS,),
    sampler=SAMPLER,
    n_trials=N_TRIALS,
    cache_arrays=False,
    seed=SEED,
    score_direction="minimize",
    polars_engine=POLARS_ENGINE,
)
pipeline

Pipeline(rolling_dates=[['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09'], ['2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'], ['2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08'], ['2026-05-11', '2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21', '2026-05-22']], test_dates=['2026-05-26', '2026-05-27', '2026-05-28', '2026-05-29'], adapter=TorchAdapter(module_builder=<function build_mlp at 0x76be1d32fce0>, loss_fn=<function torch_pinball_loss at 0x76be1d32f9c0>, optimizer_builder=<function build_optimizer at 0x76be1d32d260>, epochs=5, batch_size=8192, device='cuda', distributed=False, streaming=True), target=

In [11]:
ROLLING_DATES[-1][:1]

['2026-05-11']

In [ ]:
train_result = pipeline.train(verbose=2)
train_result

[I 2026-07-03 16:20:10,794] A new study created in memory with name: no-name-fa3e439e-6dc7-4a24-bea2-f0a8586f4795


======== Optuna study created. Launching optimization.
======== running params {'hidden_layers': 2, 'activation': 'relu', 'dropout': 0.0, 'lr': 0.0005475037105536969, 'weight_decay': 0.0005210986937158467, 'seed': 7, 'hidden_units_l1': 32, 'hidden_units_l2': 256}
======== fold: 0, with train = ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09'] and val = ['2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24']


Loading data: 7.08Mrow [00:13, 515krow/s]


======== Torch Adapter -- Epoch 0


Loading data: 7.08Mrow [00:14, 478krow/s] 
Loading data: 85.1Mrow [00:39, 2.18Mrow/s]


...

Loading data: 7.08Mrow [00:14, 480krow/s] 
Loading data: 85.1Mrow [00:39, 2.13Mrow/s]


======== Torch Adapter -- train loss = 0.7701143232439089, val loss = 0.4317819400007897


Loading data: 85.1Mrow [00:41, 2.06Mrow/s]


======== loss = 0.43173618606228575, running average = 0.43173618606228575
======== fold: 1, with train = ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'] and val = ['2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08']
======== Torch Adapter -- Epoch 0


Loading data: 10.8Mrow [00:23, 463krow/s] 
Loading data: 27.8Mrow [00:13, 3.31Mrow/s]

In [ ]:
model = pipeline.get_model()
best_hidden_sizes = getattr(model.module if hasattr(model, "module") else model, "_hidden_sizes", None)
n_params = sum(param.numel() for param in model.parameters())
best_hidden_sizes, n_params, pipeline.best_params

In [ ]:
rmse_result = pipeline.test(median_quantile(rmse))
rmse_result

In [ ]:
pnl_result = pipeline.test(
    get_quantile_pnl(
        q_buy=QUANTILES.index(0.1),
        q_sell=QUANTILES.index(0.9),
        thd_buy=0.0,
        thd_sell=0.0,
    )
)
pnl_result

In [ ]:
pnl_threshold_result = pipeline.test(
    get_quantile_pnl(
        q_buy=QUANTILES.index(0.1),
        q_sell=QUANTILES.index(0.9),
        thd_buy=0.4,
        thd_sell=-0.4,
    )
)
pnl_threshold_result

In [ ]:
y_pred_q = pnl_threshold_result["y_pred"]
print(np.mean(y_pred_q, axis=0), np.std(y_pred_q, axis=0))
_ = plt.hist(y_pred_q, bins=100, log=True, density=False, label=[f"q={q}" for q in QUANTILES])
plt.legend()
plt.xlabel("prediction")
plt.ylabel("count")

In [ ]:
model = pipeline.model